# Data cleaning

Prepare the South German Credit dataset without changing source values or row order. This notebook validates the schema, translates column names, and adds a readable target label.

## Loading the data

Locate the repository from the current directory or its parents. For a kernel started outside the repository, set `CREDIT_RISK_PROJECT_ROOT` to the checkout directory. Read the whitespace-delimited source without sorting or filtering records.

In [1]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_project_root():
    configured_root = os.environ.get("CREDIT_RISK_PROJECT_ROOT")
    candidates = (
        [Path(configured_root).expanduser().resolve()]
        if configured_root
        else [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    )
    for candidate in candidates:
        if (candidate / "data/raw/SouthGermanCredit.asc").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError(
        "Cannot locate the repository. Set CREDIT_RISK_PROJECT_ROOT to its directory."
    )


project_root = find_project_root()
raw_path = project_root / "data/raw/SouthGermanCredit.asc"
output_path = project_root / "data/processed/south_german_credit_clean.csv"
df = pd.read_csv(raw_path, sep=r"\s+")
print(f"Loaded {df.shape[0]:,} rows and {df.shape[1]} columns.")

Loaded 1,000 rows and 21 columns.


## Renaming columns

Check the exact raw schema before translating the German names. Reject missing or unexpected columns so source changes cannot silently pass through cleaning.

In [2]:
# Rename the columns 
column_names = {
    "laufkont": "checking_status",
    "laufzeit": "duration_months",
    "moral": "credit_history",
    "verw": "purpose",
    "hoehe": "credit_amount",
    "sparkont": "savings",
    "beszeit": "employment_duration",
    "rate": "installment_rate",
    "famges": "personal_status_sex",
    "buerge": "other_debtors",
    "wohnzeit": "residence_duration",
    "verm": "property",
    "alter": "age",
    "weitkred": "other_installment_plans",
    "wohn": "housing",
    "bishkred": "number_credits",
    "beruf": "job",
    "pers": "people_liable",
    "telef": "telephone",
    "gastarb": "foreign_worker",
    "kredit": "credit_risk"
}


expected_raw_columns = list(column_names)
missing_columns = set(expected_raw_columns) - set(df.columns)
unexpected_columns = set(df.columns) - set(expected_raw_columns)
if missing_columns or unexpected_columns:
    raise ValueError(
        f"Unexpected raw schema: missing={sorted(missing_columns)}, "
        f"unexpected={sorted(unexpected_columns)}"
    )

df = df.rename(columns=column_names, errors="raise")
expected_clean_columns = [column_names[name] for name in expected_raw_columns]
if list(df.columns) != expected_clean_columns:
    raise ValueError("Column renaming did not produce the expected names and order.")

## Validating the dataset

The source target must contain only `0` (Bad) or `1` (Good), with no missing values. These checks stop execution before any export.

In [3]:
allowed_target_values = {0, 1}
if df["credit_risk"].isna().any() or not df["credit_risk"].isin(allowed_target_values).all():
    raise ValueError("credit_risk must contain only 0 (Bad) and 1 (Good).")

## Target creation

Retain the original numeric target and add a readable label: **1 = Good, 0 = Bad**. The label is derived from the target and must be excluded from future model predictors.

In [4]:
df["credit_risk_label"] = df["credit_risk"].map({1: "Good", 0: "Bad"})
if df["credit_risk_label"].isna().any():
    raise ValueError("Target label creation produced missing values.")
if df.isna().any().any():
    raise ValueError("The final dataset contains missing values.")
print("Schema, target, renaming, labels, and missing-value checks passed.")

Schema, target, renaming, labels, and missing-value checks passed.


## Summary inspection

Separate measurements from category codes using the South German Credit documentation. Ordinal variables represent ordered categories, often discretized ranges; their codes are not literal measurements. `people_liable` is a binary category rather than an exact dependent count. These groups describe semantics without converting any stored values. Inspect quantitative summaries and category frequencies separately.

In [5]:
variable_groups = {
    "quantitative": ["duration_months", "credit_amount", "age"],
    "ordinal": [
        "employment_duration", "installment_rate", "residence_duration",
        "property", "number_credits", "job",
    ],
    "nominal": [
        "checking_status", "credit_history", "purpose", "savings",
        "personal_status_sex", "other_debtors", "other_installment_plans", "housing",
    ],
    "binary/target": [
        "people_liable", "telephone", "foreign_worker", "credit_risk", "credit_risk_label",
    ],
}

grouped_columns = [column for columns in variable_groups.values() for column in columns]
if len(grouped_columns) != len(set(grouped_columns)) or set(grouped_columns) != set(df.columns):
    raise ValueError("Variable groups must cover every column exactly once.")

print(f"Final shape: {df.shape}; duplicate rows: {df.duplicated().sum()}")
display(df.head())
display(df[variable_groups["quantitative"]].describe())
for group, columns in variable_groups.items():
    if group != "quantitative":
        print(f"\n{group}")
        for column in columns:
            print(f"{column}: {df[column].value_counts(dropna=False).sort_index().to_dict()}")

Final shape: (1000, 22); duplicate rows: 0


,checking_status,duration_months,credit_history,purpose,credit_amount,savings,employment_duration,installment_rate,personal_status_sex,other_debtors,...,age,other_installment_plans,housing,number_credits,job,people_liable,telephone,foreign_worker,credit_risk,credit_risk_label
0,1,18,4,2,1049,1,2,4,2,1,...,21,3,1,1,3,2,1,2,1,Good
1,1,9,4,0,2799,1,3,2,3,1,...,36,3,1,2,3,1,1,2,1,Good
2,2,12,2,9,841,2,4,2,2,1,...,23,3,1,1,2,2,1,2,1,Good
3,1,12,4,0,2122,1,3,3,3,1,...,39,3,1,2,2,1,1,1,1,Good
4,1,12,4,0,2171,1,3,4,3,1,...,38,1,2,2,2,2,1,1,1,Good


,duration_months,credit_amount,age
count,1000.000000,1000.00000,1000.00000
mean,20.903000,3271.24800,35.54200
std,12.058814,2822.75176,11.35267
min,4.000000,250.00000,19.00000
25%,12.000000,1365.50000,27.00000
50%,18.000000,2319.50000,33.00000
75%,24.000000,3972.25000,42.00000
max,72.000000,18424.00000,75.00000



ordinal
employment_duration: {1: 62, 2: 172, 3: 339, 4: 174, 5: 253}
installment_rate: {1: 136, 2: 231, 3: 157, 4: 476}
residence_duration: {1: 130, 2: 308, 3: 149, 4: 413}
property: {1: 282, 2: 232, 3: 332, 4: 154}
number_credits: {1: 633, 2: 333, 3: 28, 4: 6}
job: {1: 22, 2: 200, 3: 630, 4: 148}

nominal
checking_status: {1: 274, 2: 269, 3: 63, 4: 394}
credit_history: {0: 40, 1: 49, 2: 530, 3: 88, 4: 293}
purpose: {0: 234, 1: 103, 2: 181, 3: 280, 4: 12, 5: 22, 6: 50, 8: 9, 9: 97, 10: 12}
savings: {1: 603, 2: 103, 3: 63, 4: 48, 5: 183}
personal_status_sex: {1: 50, 2: 310, 3: 548, 4: 92}
other_debtors: {1: 907, 2: 41, 3: 52}
other_installment_plans: {1: 139, 2: 47, 3: 814}
housing: {1: 179, 2: 714, 3: 107}

binary/target
people_liable: {1: 155, 2: 845}
telephone: {1: 596, 2: 404}
foreign_worker: {1: 37, 2: 963}
credit_risk: {0: 300, 1: 700}
credit_risk_label: {'Bad': 300, 'Good': 700}


## Exporting the cleaned dataset

Repeat the validation at the export boundary to guard against interactive edits. Create the processed directory if needed and write without the dataframe index. Preserve all original rows, their order, and their values.

In [6]:
if list(df.columns) != expected_clean_columns + ["credit_risk_label"]:
    raise ValueError("Final columns do not match the expected export schema.")
if not df["credit_risk"].isin(allowed_target_values).all():
    raise ValueError("The final target contains invalid values.")
if df["credit_risk_label"].isna().any():
    raise ValueError("The final labels contain missing values.")
if not df["credit_risk_label"].equals(df["credit_risk"].map({1: "Good", 0: "Bad"})):
    raise ValueError("The final labels do not match the numeric target.")
if df.isna().any().any():
    raise ValueError("The final dataset contains missing values.")

output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)
print(f"Saved {df.shape[0]:,} rows and {df.shape[1]} columns to {output_path}")

Saved 1,000 rows and 22 columns to /Users/havuminhthu/Documents/finance_proj/credit-risk-analytics/data/processed/south_german_credit_clean.csv


## Cleaning summary

Cleaning is complete: source values and record order are preserved, columns have readable names, and the target has a validated label. The supplied data contain 1,000 records, no missing values, and no duplicate rows. EDA, SQL analysis, modeling, and dashboard/visualization are planned stages and have not been implemented.

Source and category semantics: [UCI South German Credit](https://www.archive.ics.uci.edu/dataset/573/south%2Bgerman%2Bcredit%2Bupdate).